In [69]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("ChicagoTaxiPipeline").getOrCreate()

In [70]:
file_url = "/home/ouyassine/Documents/projects/taxi pipeline/data/raw_data/chicago_taxi_trips_2020_2021.csv"

dataset = spark.read.csv(file_url, header=True, inferSchema=True)



In [71]:
dataset.show(5)

dataset.printSchema()

+--------------------+--------------------+--------------------+-------------------+------------+----------+-------------------+--------------------+---------------------+----------------------+----+----+-----+------+----------+------------+--------------------+------------------------+-------------------------+------------------------+-------------------------+--------------------------+-------------------------+
|             trip_id|             taxi_id|trip_start_timestamp| trip_end_timestamp|trip_seconds|trip_miles|pickup_census_tract|dropoff_census_tract|pickup_community_area|dropoff_community_area|fare|tips|tolls|extras|trip_total|payment_type|             company|pickup_centroid_latitude|pickup_centroid_longitude|pickup_centroid_location|dropoff_centroid_latitude|dropoff_centroid_longitude|dropoff_centroid_location|
+--------------------+--------------------+--------------------+-------------------+------------+----------+-------------------+--------------------+---------------

In [72]:
print("Number of rows: ", dataset.count())

Number of rows:  4560000


In [73]:
print("columns: ", dataset.columns)

columns:  ['trip_id', 'taxi_id', 'trip_start_timestamp', 'trip_end_timestamp', 'trip_seconds', 'trip_miles', 'pickup_census_tract', 'dropoff_census_tract', 'pickup_community_area', 'dropoff_community_area', 'fare', 'tips', 'tolls', 'extras', 'trip_total', 'payment_type', 'company', 'pickup_centroid_latitude', 'pickup_centroid_longitude', 'pickup_centroid_location', 'dropoff_centroid_latitude', 'dropoff_centroid_longitude', 'dropoff_centroid_location']


# Data cleaning

In [74]:
from pyspark.sql.functions import col, sum

# how many null values are in each column
null_counts = dataset.select([sum(dataset[col].isNull().cast("int")).alias(col) for col in dataset.columns])
null_counts.show()

#percetange of null values in each column
total_rows = dataset.count()
null_percentage = null_counts.select([
    (col(column_name) / total_rows * 100).alias(column_name)
    for column_name in null_counts.columns
])
null_percentage.show()

+-------+-------+--------------------+------------------+------------+----------+-------------------+--------------------+---------------------+----------------------+----+----+-----+------+----------+------------+-------+------------------------+-------------------------+------------------------+-------------------------+--------------------------+-------------------------+
|trip_id|taxi_id|trip_start_timestamp|trip_end_timestamp|trip_seconds|trip_miles|pickup_census_tract|dropoff_census_tract|pickup_community_area|dropoff_community_area|fare|tips|tolls|extras|trip_total|payment_type|company|pickup_centroid_latitude|pickup_centroid_longitude|pickup_centroid_location|dropoff_centroid_latitude|dropoff_centroid_longitude|dropoff_centroid_location|
+-------+-------+--------------------+------------------+------------+----------+-------------------+--------------------+---------------------+----------------------+----+----+-----+------+----------+------------+-------+----------------------

+-------+-------+--------------------+--------------------+-------------------+--------------------+-------------------+--------------------+---------------------+----------------------+--------------------+--------------------+--------------------+--------------------+--------------------+------------+-------+------------------------+-------------------------+------------------------+-------------------------+--------------------------+-------------------------+
|trip_id|taxi_id|trip_start_timestamp|  trip_end_timestamp|       trip_seconds|          trip_miles|pickup_census_tract|dropoff_census_tract|pickup_community_area|dropoff_community_area|                fare|                tips|               tolls|              extras|          trip_total|payment_type|company|pickup_centroid_latitude|pickup_centroid_longitude|pickup_centroid_location|dropoff_centroid_latitude|dropoff_centroid_longitude|dropoff_centroid_location|
+-------+-------+--------------------+--------------------+-----

## drop pickup_census_tract and dropoff census tract cuz 61% of  data is null

In [75]:
#drop two columns that are mostly null
dataset = dataset.drop("pickup_census_tract", "dropoff_census_tract")
print("Number of rows: ", dataset.count())

Number of rows:  4560000


In [76]:
#check for duplicates
if dataset.count() != dataset.dropDuplicates().count():
    print("There are duplicates in the dataset")


# drop unnnessary columns

In [77]:
dataset = dataset.drop("trip_end_timestamp", "pickup_centroid_latitude", "pickup_centroid_location", "dropoff_centroid_location", "pickup_centroid_longitude", "dropoff_centroid_latitude", "dropoff_centroid_longitude")

In [78]:
#validate all numbers are positive
from pyspark.sql.functions import col
dataset = dataset.filter(
    (col("trip_miles") >= 0) &
    (col("trip_seconds") >= 0) &
    (col("fare") >= 0) &
    (col("tips") >= 0) &
    (col("tolls") >= 0) &
    (col("extras") >= 0) &
    (col("trip_total") >= 0)
)
print("Number of rows: ", dataset.count())

Number of rows:  4557703


In [79]:
#enforce that trip_total = fare + tips + tolls + extras



from pyspark.sql.functions import col, abs

dataset = dataset.withColumn(
    "calculated_total",
    col("fare") + col("tips") + col("tolls") + col("extras")
)

dataset = dataset.withColumn(
    "total_difference",
    abs(col("trip_total") - col("calculated_total"))
)

df = dataset.filter(col("total_difference") > 0)
df.show(10)

print("Number of rows: ", dataset.count())
print("Number of rows where trip_total does not equal fare + tips + tolls + extras: ", df.count())

+--------------------+--------------------+--------------------+------------+----------+---------------------+----------------------+-----+----+-----+------+----------+------------+--------------------+----------------+----------------+
|             trip_id|             taxi_id|trip_start_timestamp|trip_seconds|trip_miles|pickup_community_area|dropoff_community_area| fare|tips|tolls|extras|trip_total|payment_type|             company|calculated_total|total_difference|
+--------------------+--------------------+--------------------+------------+----------+---------------------+----------------------+-----+----+-----+------+----------+------------+--------------------+----------------+----------------+
|451beabd5dd8ffc54...|a572ffd405aa83dbb...| 2020-01-01 00:00:00|         960|       2.4|                    8|                     8|10.75| 0.0|  0.0|   1.0|     12.25| Credit Card|    Medallion Leasin|           11.75|             0.5|
|60da9b02b3297dc58...|155ffe17bc32e7c3b...| 2020-01-

Number of rows:  4557703


Number of rows where trip_total does not equal fare + tips + tolls + extras:  872409


In [80]:
dataset.printSchema()

root
 |-- trip_id: string (nullable = true)
 |-- taxi_id: string (nullable = true)
 |-- trip_start_timestamp: timestamp (nullable = true)
 |-- trip_seconds: integer (nullable = true)
 |-- trip_miles: double (nullable = true)
 |-- pickup_community_area: integer (nullable = true)
 |-- dropoff_community_area: integer (nullable = true)
 |-- fare: double (nullable = true)
 |-- tips: double (nullable = true)
 |-- tolls: double (nullable = true)
 |-- extras: double (nullable = true)
 |-- trip_total: double (nullable = true)
 |-- payment_type: string (nullable = true)
 |-- company: string (nullable = true)
 |-- calculated_total: double (nullable = true)
 |-- total_difference: double (nullable = true)



In [81]:
# enforce schema for the dataset
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, TimestampType

dataset_silver = dataset.select(
    col("trip_id").cast(StringType()).alias("trip_id"),
    col("taxi_id").cast(StringType()).alias("taxi_id"),
    col("trip_start_timestamp").cast(TimestampType()).alias("trip_start_timestamp"),
    col("trip_seconds").cast(IntegerType()).alias("trip_seconds"),
    col("trip_miles").cast(DoubleType()).alias("trip_miles"),
    col("pickup_community_area").cast(IntegerType()).alias("pickup_community_area"),
    col("dropoff_community_area").cast(IntegerType()).alias("dropoff_community_area"),
    col("fare").cast(DoubleType()).alias("fare"),
    col("tips").cast(DoubleType()).alias("tips"),
    col("tolls").cast(DoubleType()).alias("tolls"),
    col("extras").cast(DoubleType()).alias("extras"),
    col("trip_total").cast(DoubleType()).alias("trip_total"),
    col("payment_type").cast(StringType()).alias("payment_type"),
    col("company").cast(StringType()).alias("company"),
)   


In [ ]:
print("Number of rows: ", dataset_silver.count())

Number of rows:  4557703


: 

In [ ]:
# based on offial documentation, trips with pickup area null are outside chicago city limits
dataset_silver = dataset_silver.withColumn("pickup_outside_chicago", col("pickup_community_area").isNull())
dataset_silver = dataset_silver.withColumn("dropoff_outside_chicago", col("dropoff_community_area").isNull())